In [1]:

# Upgrade pip first
!pip install --upgrade pip

# Core ML libraries
!pip install torch==2.1.0+cu121 torchvision==0.16.1+cu121 torchaudio==2.1.1 --extra-index-url https://download.pytorch.org/whl/cu121

# Hugging Face and tokenizer tools
!pip install transformers==4.34.0 sentencepiece tokenizers safetensors

# Efficient fine-tuning
!pip install peft bitsandbytes accelerate einops

# Dataset and evaluation
!pip install datasets evaluate

# Experiment tracking (optional but recommended)
!pip install wandb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torch==2.1.0+cu121 (from versions: 2.2.0, 2.2.0+cu121, 2.2.1, 2.2.1+cu121, 2.2.2, 2.2.2+cu121, 2.3.0, 2.3.0+cu121, 2.3.1, 2.3.1+cu121, 2.4.0, 2.4.0+cu121, 2.4.1, 2.4.1+cu121, 2.5.0, 2.5.0+cu121, 2.5.1, 2.5.1+cu121, 2.6.0, 2.7.0, 2.7.1, 2.8.0)
ERROR: No matching distribution found for torch==2.1.0+cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 56.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 64.3 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.34.4
    Uninstalling huggingface-hub-0.34.4:
      Successfully uninstalled huggingface-h

### Mistral Experimentation for Finance domain finetuning

In [2]:

from datasets import load_dataset, DatasetDict

# Load the JSONL file
train_ds = load_dataset("json", data_files="/content/finance_pc_train.jsonl", split="train")

# display the first row
print(train_ds[0])

# Check the column names
print(train_ds.column_names)


{'question': "What services does Iron Mountain provide to protect organizations' information and reduce storage costs?", 'context': 'Iron Mountain helps organizations protect their information and reduce storage costs by storing physical records and data backup media, offering information management solutions, and providing data center space.', 'answer': 'Iron Mountain provides services such as storing physical records and data backup media, offering information management solutions, and providing data center space for enterprise-class colocation and hyperscale deployments.'}
['question', 'context', 'answer']


In [3]:
full_ds = load_dataset("json", data_files="/content/finance_pc_train.jsonl", split="train")

# Split 10% for validation and 90% for the training
split_ds = full_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

print("Train samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Columns:", train_ds.column_names)
print("Sample:", train_ds[0])

Train samples: 900
Validation samples: 100
Columns: ['question', 'context', 'answer']
Sample: {'question': 'What is the significance of Note 13 in the context of legal proceedings described in the Annual Report on Form 10-K?', 'context': 'For a description of our significant pending legal proceedings, see Note 13 titled Commitments and Contingencies - Legal Proceedings of the Notes to Consolidated Financial Statements included in Part II, Item 8 of this Annual Report on Form 10-K.', 'answer': "Note 13 is significant because it contains a detailed description of the company's significant pending legal proceedings."}


In [4]:
from transformers import AutoTokenizer, default_data_collator
import math

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_user_text(q: str, c: str) -> str:
    q = (q or "").strip()
    c = (c or "").strip()
    return f"{q}\n\nContext:\n{c}" if c else q

def preprocess(example):
    # chat strings
    user = build_user_text(example["question"], example["context"])
    messages_full = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": (example["answer"] or "").strip()},
    ]
    messages_prompt_only = [{"role": "user", "content": user}]

    prompt_text = tokenizer.apply_chat_template(
        messages_prompt_only, tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        messages_full, tokenize=False, add_generation_prompt=False
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(full_text,  add_special_tokens=False)["input_ids"]

    # guard rails
    if not full_ids or len(full_ids) < len(prompt_ids):
        return {"drop": True}

    # ensure prefix alignment or find it
    if full_ids[:len(prompt_ids)] != prompt_ids:
        found = -1
        for i in range(0, max(0, len(full_ids) - len(prompt_ids)) + 1):
            if full_ids[i:i+len(prompt_ids)] == prompt_ids:
                found = i
                break
        if found == -1:
            return {"drop": True}
        split_at = found + len(prompt_ids)
    else:
        split_at = len(prompt_ids)

    labels = [-100] * split_at + full_ids[split_at:]

    # truncate
    input_ids = full_ids[:MAX_LEN]
    labels    = labels[:MAX_LEN]
    attention_mask = [1] * len(input_ids)

    # drop if answer fully truncated
    if not input_ids or all(l == -100 for l in labels):
        return {"drop": True}

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "drop": False,
    }

cols_to_remove = list(train_ds.column_names)  # ['question','context','answer']

train_ds_tok = train_ds.map(preprocess, remove_columns=cols_to_remove, desc="tokenize/train")
train_ds_tok = train_ds_tok.filter(lambda ex: ex["drop"] is False, desc="filter/train")
train_ds_tok = train_ds_tok.remove_columns(["drop"])

val_ds_tok = val_ds.map(preprocess, remove_columns=cols_to_remove, desc="tokenize/val")
val_ds_tok = val_ds_tok.filter(lambda ex: ex["drop"] is False, desc="filter/val")
val_ds_tok = val_ds_tok.remove_columns(["drop"])

# extra safety: remove any empties
train_ds_tok = train_ds_tok.filter(lambda ex: len(ex["input_ids"]) > 0 and any(l != -100 for l in ex["labels"]))
val_ds_tok   = val_ds_tok.filter(lambda ex: len(ex["input_ids"]) > 0 and any(l != -100 for l in ex["labels"]))

# quick sanity
print("train/val sizes:", len(train_ds_tok), len(val_ds_tok))
print("sample lens:", len(train_ds_tok[0]["input_ids"]), len(train_ds_tok[0]["labels"]))




train/val sizes: 900 100
sample lens: 116 116


In [5]:
## QLoRA finetune with mistral model loading

import os, math, torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = "mistral7b_v03_qlora_finance"

def has_flash_attn():
    try:
        import flash_attn  # noqa
        return True
    except Exception:
        return False

# 4-bit NF4 quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                          # nf4 better than other dtatatypes
    bnb_4bit_compute_dtype=torch.bfloat16
        if torch.cuda.is_available() else torch.float16,
    bnb_4bit_use_double_quant=True                     # to reduce memory
)

# Load the base model quantized with nf4 datatype
attn_impl = "flash_attention_2" if has_flash_attn() else "eager"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    attn_implementation=attn_impl,
)

# Important for gradient checkpointing later
model.config.use_cache = False

# LoRA config — fairly beefy but still memory-friendly
lora_config = LoraConfig(
    r=32,                         # 16 performeed better in my previous experiment with LLaMA model
    lora_alpha=32,                # scaling
    lora_dropout=0.05,            # regularization
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[              # attention + MLP finetuning from my previous observation
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_rslora=True,
)

model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

trainable params: 83,886,080 || all params: 7,331,909,632 || trainable%: 1.1441
None


In [6]:
import wandb
wandb.login()
os.environ["WANDB_PROJECT"] = "mistral-finance-sft"
RUN_NAME = "mistral7b_v03_qlora_teacherforcing"


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhi1199 (abhi1199-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
import os, math, torch, wandb
from transformers import TrainingArguments, Trainer, TrainerCallback

# W&B
wandb.login()
os.environ["WANDB_PROJECT"] = "mistral-finance-sft"
RUN_NAME = "mistral7b_v03_qlora_teacherforcing_2"

class PerplexityPrinter(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        if "loss" in logs:
            loss = float(logs["loss"])
            ppl = math.exp(loss) if loss < 20 else float("inf")
            print(f"[train] step {int(state.global_step)} - loss {loss:.4f} - ppl {ppl:.2f}")
            wandb.log({"train/loss": loss, "train/perplexity": ppl, "step": int(state.global_step)})

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics or "eval_loss" not in metrics: return
        eval_loss = float(metrics["eval_loss"])
        eval_ppl = math.exp(eval_loss) if eval_loss < 20 else float("inf")
        print(f"[eval]  step {int(state.global_step)} - eval_loss {eval_loss:.4f} - eval_ppl {eval_ppl:.2f}")
        wandb.log({"eval/loss": eval_loss, "eval/perplexity": eval_ppl, "step": int(state.global_step)})


In [8]:
args = TrainingArguments(
    output_dir="mistral7b_v03_qlora_out",
    run_name=RUN_NAME,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_grad_norm=1.0,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to=["wandb"],
    remove_unused_columns=False,
)

In [9]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)



if hasattr(model, "_orig_mod"):
    # unwrap compiled model so Trainer accepts it for PEFT fine-tuning
    model = model._orig_mod


# Trainer
from transformers import TrainingArguments, Trainer
import math, wandb, torch

args = TrainingArguments(
    output_dir="mistral7b_v03_qlora_out",
    run_name="mistral7b_v03_qlora_teacherforcing",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_grad_norm=1.0,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to=["wandb"],
    remove_unused_columns=False,
)


class PerplexityPrinter(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        if "loss" in logs:
            loss = float(logs["loss"])
            ppl = math.exp(loss) if loss < 20 else float("inf")
            print(f"[train] step {int(state.global_step)} - loss {loss:.4f} - ppl {ppl:.2f}")
            wandb.log({"train/loss": loss, "train/perplexity": ppl, "step": int(state.global_step)})

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics or "eval_loss" not in metrics: return
        eval_loss = float(metrics["eval_loss"])
        eval_ppl = math.exp(eval_loss) if eval_loss < 20 else float("inf")
        print(f"[eval]  step {int(state.global_step)} - eval_loss {eval_loss:.4f} - eval_ppl {eval_ppl:.2f}")
        wandb.log({"eval/loss": eval_loss, "eval/perplexity": eval_ppl, "step": int(state.global_step)})

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds_tok,
    eval_dataset=val_ds_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[PerplexityPrinter()],
)

train_result = trainer.train()


Step,Training Loss,Validation Loss


[train] step 20 - loss 0.7154 - ppl 2.05
[train] step 40 - loss 0.5551 - ppl 1.74
[train] step 60 - loss 0.4817 - ppl 1.62
[train] step 80 - loss 0.1689 - ppl 1.18
[train] step 100 - loss 0.1639 - ppl 1.18
[train] step 120 - loss 0.1428 - ppl 1.15
[train] step 140 - loss 0.0599 - ppl 1.06
[train] step 160 - loss 0.0440 - ppl 1.04


In [10]:
trainer.save_model()
tokenizer.save_pretrained(args.output_dir)



[eval]  step 171 - eval_loss 0.6499 - eval_ppl 1.92
[final] eval_loss 0.6499 - eval_ppl 1.92
{'eval_loss': 0.649901270866394, 'eval_runtime': 11.3872, 'eval_samples_per_second': 8.782, 'eval_steps_per_second': 4.391, 'epoch': 3.0}


In [10]:
# --- Final eval + print + log ---
metrics = trainer.evaluate()
if "eval_loss" in metrics:
    try:
        eval_ppl = math.exp(metrics["eval_loss"])
    except OverflowError:
        eval_ppl = float("inf")
    print(f"[final] eval_loss {metrics['eval_loss']:.4f} - eval_ppl {eval_ppl:.2f}")
    wandb.log({"final/eval_loss": metrics["eval_loss"], "final/eval_perplexity": eval_ppl})
print(metrics)

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch, os

# --- Paths ---
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_DIR = "./mistral7b_v03_qlora_out"

HF_TOKEN = os.getenv("HF_TOKEN", None)

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Quantized base model (QLoRA-style) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    token=HF_TOKEN,

)

# --- Attach your LoRA adapter ---
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
model.config.use_cache = True

print("Mistral base + LoRA adapter loaded.")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Mistral base + LoRA adapter loaded.


In [12]:
def ask_mistral(context: str, question: str,
                max_new_tokens: int = 200,
                temperature: float = 0.7,
                top_p: float = 0.9,
                repetition_penalty: float = 1.1,
                do_sample: bool = True) -> str:
    #context first, then question
    messages = [
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}"}
    ]

    # to generate
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Keep only new tokens ot the prompt
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer


In [13]:
# test with first sample question
context = "Open Value agreements are a simple, cost-effective way to acquire the latest Microsoft technology. These agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under Open Value agreements, organizations can elect to purchase perpetual licenses or subscribe to licenses and SA is included."
question = "What type of organizations is the Open Value agreements designed for and what licenses does it include?"
answer = ask_mistral(context, question)
print("Answer:", answer)


Answer: The Open Value agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a period of three years. They include options to purchase perpetual licenses or subscribe to licenses and SA is included.


In [17]:
context = ""
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_mistral(context, question)
print("Answer:", answer)


Answer: AT&T's wireless segment delivers nationwide wireless service and equipment.


In [15]:

context = "The Company allocates the transaction price to each performance obligation on a relative SSP basis. Judgment is required to determine the SSP for each distinct performance obligation. The Company determines SSP by considering its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices."
question = "What is the basis for the Company to determine the Standalone Selling Price (SSP) for each distinct performance obligation in contracts with multiple performance obligations?"
answer = ask_mistral(context, question)
print("Answer:", answer)


Answer: The Company determines the SSP based on its overall pricing objectives and market conditions, including discounting practices, the size and volume of transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, and historical and current sales and contract prices.


In [16]:
context = "As the rate implicit in the lease is rarely readily determinable, Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date."
question = "What discount rate does Delta Air Lines use for lease payments when the rate implicit in the lease is not readily determinable?"
answer = ask_mistral(context, question)
print("Answer:", answer)

Answer: Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date.
